In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/nishi-kasat/RI.git
%cd RI

Cloning into 'RI'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 12 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 66.54 KiB | 33.27 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/RI


In [4]:
!pip install -q torch torchvision timm scikit-learn pandas matplotlib tensorboard opencv-python

In [6]:
ZIP_PATH = "/content/drive/MyDrive/Research-Paper/Dataset/state-farm-distracted-driver-detection.zip"
!mkdir -p /content/dataset
!unzip -q "$ZIP_PATH" -d /content/dataset

# Milestone 2: MobileViT-XXS Baseline Training
This notebook implements the baseline training pipeline for distracted driver detection using the MobileViT-XXS architecture.

In [ ]:
import os
import time
import json
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Tuple, List, Dict, Any, Optional
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
import timm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

# --- Configuration ---
class Config:
    SEED = 42
    DATA_DIR = Path('/content/dataset')
    CSV_PATH = DATA_DIR / 'driver_imgs_list.csv'
    WEIGHTS_DIR = Path('weights/mobilevit')
    OUTPUT_DIR = Path('outputs/training')
    FIGURES_DIR = Path('outputs/figures')
    LOG_DIR = Path('outputs/tensorboard/mobilevit')

    IMAGE_SIZE = 224
    BATCH_SIZE = 32
    EPOCHS = 20
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    PATIENCE = 5
    NUM_CLASSES = 10
    MODEL_NAME = 'mobilevit_xxs'

# Initialize paths
for p in [Config.WEIGHTS_DIR, Config.OUTPUT_DIR, Config.FIGURES_DIR, Config.LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(Config.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
class DistractedDriverDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: Path, transform: Optional[Any] = None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(df['classname'].unique())
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        row = self.df.iloc[idx]
        img_path = self.root_dir / 'imgs' / 'train' / row['classname'] / row['img']
        image = Image.open(img_path).convert('RGB')
        label = self.class_to_idx[row['classname']]
        if self.transform:
            image = self.transform(image)
        return image, label

def get_dataloaders() -> Tuple[DataLoader, DataLoader]:
    df = pd.read_csv(Config.CSV_PATH)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=Config.SEED)
    train_idx, val_idx = next(gss.split(df, groups=df['subject']))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    train_transform = transforms.Compose([
        transforms.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.2, 0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        transforms.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_ds = DistractedDriverDataset(train_df, Config.DATA_DIR, train_transform)
    val_ds = DistractedDriverDataset(val_df, Config.DATA_DIR, val_transform)

    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)

    return train_loader, val_loader

In [ ]:
class MobileViTTrainer:
    def __init__(self, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=Config.EPOCHS)
        self.scaler = torch.cuda.amp.GradScaler()
        self.writer = SummaryWriter(Config.LOG_DIR)

        self.history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
        self.best_acc = 0.0
        self.patience_counter = 0

    def train_epoch(self, epoch: int) -> Tuple[float, float]:
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [Train]")

        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            self.optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            pbar.set_postfix({'loss': loss.item(), 'acc': 100.*correct/total})

        return running_loss/total, 100.*correct/total

    @torch.no_grad()
    def validate(self) -> Tuple[float, float]:
        self.model.eval()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in self.val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        return running_loss/total, 100.*correct/total

    def fit(self):
        start_epoch = 0
        latest_path = Config.WEIGHTS_DIR / 'latest_model.pth'
        if latest_path.exists():
            checkpoint = torch.load(latest_path)
            self.model.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            print(f"Resuming from epoch {start_epoch}")

        for epoch in range(start_epoch, Config.EPOCHS):
            t_loss, t_acc = self.train_epoch(epoch)
            v_loss, v_acc = self.validate()
            self.scheduler.step()

            self.history['train_loss'].append(t_loss); self.history['train_acc'].append(t_acc)
            self.history['val_loss'].append(v_loss); self.history['val_acc'].append(v_acc)

            self.writer.add_scalar('Loss/Train', t_loss, epoch)
            self.writer.add_scalar('Accuracy/Train', t_acc, epoch)
            self.writer.add_scalar('Loss/Val', v_loss, epoch)
            self.writer.add_scalar('Accuracy/Val', v_acc, epoch)

            torch.save({'epoch': epoch, 'model_state_dict': self.model.state_dict(), 'optimizer_state_dict': self.optimizer.state_dict()}, latest_path)

            if v_acc > self.best_acc:
                self.best_acc = v_acc
                torch.save(self.model.state_dict(), Config.WEIGHTS_DIR / 'best_model.pth')
                self.patience_counter = 0
            else:
                self.patience_counter += 1

            if self.patience_counter >= Config.PATIENCE:
                print("Early stopping triggered."); break

        pd.DataFrame(self.history).to_csv(Config.OUTPUT_DIR / 'training_history.csv', index=False)
        self.writer.close()

In [ ]:
# Model Building & Execution
train_loader, val_loader = get_dataloaders()
model = timm.create_model(Config.MODEL_NAME, pretrained=True, num_classes=Config.NUM_CLASSES)
trainer = MobileViTTrainer(model, train_loader, val_loader)

# To train, uncomment the following line:
# trainer.fit()

### Evaluation & Visualization
Run the following code after training to generate metrics and plots.

In [ ]:
def plot_metrics():
    history = pd.read_csv(Config.OUTPUT_DIR / 'training_history.csv')
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    ax[0].plot(history['train_loss'], label='Train'); ax[0].plot(history['val_loss'], label='Val')
    ax[0].set_title('Loss Curve'); ax[0].legend()
    ax[1].plot(history['train_acc'], label='Train'); ax[1].plot(history['val_acc'], label='Val')
    ax[1].set_title('Accuracy Curve'); ax[1].legend()
    plt.savefig(Config.FIGURES_DIR / 'training_curves.png'); plt.show()

@torch.no_grad()
def final_evaluation():
    model.load_state_dict(torch.load(Config.WEIGHTS_DIR / 'best_model.pth'))
    model.eval()
    y_true, y_pred = [], []
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        y_true.extend(labels.tolist())
        y_pred.extend(outputs.argmax(1).cpu().tolist())

    print(classification_report(y_true, y_pred))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.savefig(Config.FIGURES_DIR / 'confusion_matrix.png'); plt.show()

print("Milestone 2 Completed Successfully")